# Layer 1a — 系統異常偵測

目標：找出系統層面的異常訂單（排除已知 user burst 後）。
可能原因：SECS timeout、DB lock、queue stuck。

**改進**：先排除 Layer 1b 標記的 user_burst，避免 burst 造成的 queue 延長被誤判為系統異常。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)
PARALLELISM = 4

df = pd.read_csv('../data/orders.csv')
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format='%Y/%m/%d %I:%M:%S %p')

# Load user anomaly flags (1b runs first)
usr_flags = pd.read_csv('../data/user_anomaly_flags.csv')
df = df.merge(usr_flags, on='order_id')

print(f"Total orders: {len(df)}")
print(f"User anomalies (from 1b): {df['is_user_anomaly'].sum()}")
print(f"Unique devices: {df['device_id'].nunique()}")


## 1. 排除 user burst 後，計算 per-device IQR 閾值

使用非 user_burst 的訂單來建立 per-device baseline。
偵測維度：`device_duration_avg`、`db_duration_avg`（per-device IQR×4）。
Queue stuck：全域閾值 >500s（與 user burst 80-500s 區隔）。

In [ ]:
# Use non-burst orders to compute device-level thresholds
non_burst = df[~df['is_user_anomaly']]
IQR_MULTIPLIER = 4
QUEUE_STUCK_THRESHOLD = 500

# Per-device IQR for device and db durations
device_thresholds = non_burst.groupby('device_id').agg(
    dev_q1=('device_duration_avg_seconds', lambda x: x.quantile(0.25)),
    dev_q3=('device_duration_avg_seconds', lambda x: x.quantile(0.75)),
    db_q1=('db_duration_avg_seconds', lambda x: x.quantile(0.25)),
    db_q3=('db_duration_avg_seconds', lambda x: x.quantile(0.75)),
).reset_index()

device_thresholds['upper_device'] = device_thresholds['dev_q3'] + IQR_MULTIPLIER * (device_thresholds['dev_q3'] - device_thresholds['dev_q1'])
device_thresholds['upper_db'] = device_thresholds['db_q3'] + IQR_MULTIPLIER * (device_thresholds['db_q3'] - device_thresholds['db_q1'])

print(f"Queue stuck threshold: > {QUEUE_STUCK_THRESHOLD}s")
print(f"Device/DB: per-device Q3 + {IQR_MULTIPLIER}×IQR (computed on non-burst orders)")

# Merge thresholds back
df = df.merge(device_thresholds[['device_id', 'upper_device', 'upper_db']], on='device_id')

# Flag system anomalies (only on non-burst orders)
df['is_system_anomaly'] = (
    ~df['is_user_anomaly'] & (
        (df['queue_duration_seconds'] > QUEUE_STUCK_THRESHOLD) |
        (df['device_duration_avg_seconds'] > df['upper_device']) |
        (df['db_duration_avg_seconds'] > df['upper_db'])
    )
)

anomalies = df[df['is_system_anomaly']]
print(f"System anomalies: {len(anomalies)} / {len(df)} ({100*len(anomalies)/len(df):.1f}%)")
print(f"Devices with anomalies: {anomalies['device_id'].nunique()}")


## 2. 異常分類與 Phase Breakdown

In [ ]:
# Classify anomaly type
def classify_anomaly(row):
    reasons = []
    if row['queue_duration_seconds'] > QUEUE_STUCK_THRESHOLD:
        reasons.append('queue_stuck')
    if row['device_duration_avg_seconds'] > row['upper_device']:
        reasons.append('device_timeout')
    if row['db_duration_avg_seconds'] > row['upper_db']:
        reasons.append('db_lock')
    return ', '.join(reasons) if reasons else 'unknown'

anomalies = anomalies.copy()
anomalies['anomaly_type'] = anomalies.apply(classify_anomaly, axis=1)

# Phase breakdown
anomalies['est_queue'] = anomalies['queue_duration_seconds']
anomalies['est_db'] = anomalies['db_duration_avg_seconds'] * anomalies['file_count'] / PARALLELISM
anomalies['est_device'] = anomalies['device_duration_avg_seconds'] * anomalies['file_count'] / PARALLELISM
anomalies['est_inner'] = anomalies['inner_processing_duration_avg_seconds'] * anomalies['file_count'] / PARALLELISM

phase_cols = ['est_queue', 'est_db', 'est_device', 'est_inner']
anomalies['dominant_phase'] = anomalies[phase_cols].idxmax(axis=1).str.replace('est_', '')

print("Anomaly type distribution:")
print(anomalies['anomaly_type'].value_counts().head(10))
print("\nDominant phase in anomalies:")
print(anomalies['dominant_phase'].value_counts())


## 3. Ground Truth 驗證

In [ ]:
# Validate against ground truth
labels = pd.read_csv('../data/orders_with_labels.csv')
merged = df[['order_id', 'is_system_anomaly']].merge(labels[['order_id', '_label']], on='order_id')

true_sys = merged['_label'].isin(['queue_stuck', 'device_timeout', 'db_lock'])
pred_sys = merged['is_system_anomaly']

tp = (pred_sys & true_sys).sum()
fp = (pred_sys & ~true_sys).sum()
fn = (~pred_sys & true_sys).sum()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"=== System Anomaly Validation ===")
print(f"  TP={tp}, FP={fp}, FN={fn}")
print(f"  Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")
if fp > 0:
    print(f"\nFP label distribution:")
    print(merged.loc[pred_sys & ~true_sys, '_label'].value_counts().to_string())


## 4. 圖表

In [ ]:
# Chart 1: Per-device duration boxplot (devices with highest anomaly rate)
anomaly_rate = df.groupby('device_id')['is_system_anomaly'].mean()
top_anomaly_devices = anomaly_rate.nlargest(20).index
plot_df = df[df['device_id'].isin(top_anomaly_devices)]

fig, ax = plt.subplots(figsize=(14, 6))
device_order = plot_df.groupby('device_id')['per_file_duration_avg_seconds'].median().sort_values(ascending=False).index
sns.boxplot(data=plot_df, x='device_id', y='per_file_duration_avg_seconds', order=device_order,
            flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5}, ax=ax)
ax.set_xticks(range(len(device_order)))
ax.set_xticklabels(device_order, rotation=45, ha='right', fontsize=8)
ax.set_title('Per-File Duration by Device (Top 20 by Anomaly Rate)')
ax.set_xlabel('Device ID')
ax.set_ylabel('Per-File Avg Duration (seconds)')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '1a_device_duration_boxplot.png', dpi=150)
plt.close()
print("Saved: 1a_device_duration_boxplot.png")


In [ ]:
# Chart 2: Anomaly type and dominant phase
anomaly_type_counts = anomalies['anomaly_type'].str.split(', ').explode().value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

anomaly_type_counts.plot(kind='barh', ax=axes[0], color=sns.color_palette('Set2'))
axes[0].set_title('System Anomaly Type Distribution')
axes[0].set_xlabel('Count')

phase_counts = anomalies['dominant_phase'].value_counts()
colors = {'device': '#e74c3c', 'db': '#3498db', 'queue': '#f39c12', 'inner': '#2ecc71'}
phase_counts.plot(kind='bar', ax=axes[1], color=[colors.get(x, '#95a5a6') for x in phase_counts.index])
axes[1].set_title('Dominant Phase in Anomalous Orders')
axes[1].set_xlabel('Phase')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(REPORTS_DIR / '1a_anomaly_phase_breakdown.png', dpi=150)
plt.close()
print("Saved: 1a_anomaly_phase_breakdown.png")


In [ ]:
# Chart 3: Timeline
fig, ax = plt.subplots(figsize=(14, 6))
normal = df[~df['is_system_anomaly'] & ~df['is_user_anomaly']]
ax.scatter(normal['order_created_at'], normal['total_duration_seconds'],
           alpha=0.05, s=5, color='gray', label='Normal')
ax.scatter(anomalies['order_created_at'], anomalies['total_duration_seconds'],
           alpha=0.6, s=15, c='red', label='System Anomaly')
ax.set_title('Order Duration Timeline (System Anomalies Highlighted)')
ax.set_xlabel('Order Created Time')
ax.set_ylabel('Total Duration (seconds)')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '1a_anomaly_timeline.png', dpi=150)
plt.close()
print("Saved: 1a_anomaly_timeline.png")


## 5. 匯出

In [ ]:
# Export
anomaly_flags = df[['order_id', 'is_system_anomaly']].copy()
anomaly_flags.to_csv('../data/system_anomaly_flags.csv', index=False)
print(f"Exported {anomaly_flags['is_system_anomaly'].sum()} system anomaly flags")

print(f"\n=== Layer 1a Summary ===")
print(f"Total orders: {len(df)}")
print(f"System anomalies: {len(anomalies)} ({100*len(anomalies)/len(df):.1f}%)")
print(f"Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")
